# 03 - Peek-Safe Sequential Testing
**Milestone 4**

Teams always want to look at results early. Naive peeking inflates the type-I
error. This notebook shows two correct answers:

* **group-sequential** boundaries via Lan-DeMets alpha spending (O'Brien-Fleming, Pocock),
* **always-valid p-values** via the mSPRT Gaussian mixture.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src")) and os.path.exists(
            os.path.join(d, "src", "config.py")
        ):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start)
        d = parent

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import artifact_path, raw_path, ensure_directories
ensure_directories()
FIGS = artifact_path("figures")
os.makedirs(FIGS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIGURES    :", FIGS)


## 1. Alpha-spending boundaries for 5 equally spaced looks

In [ ]:
from src.core import sequential as seq
from src.core.conclude import sequential_conclusion

b = seq.group_sequential_boundaries(n_looks=5, alpha=0.05, spending="obrien-fleming")
pd.DataFrame({
    "look": range(1, 6),
    "information_fraction": [round(t, 2) for t in b["information_times"]],
    "cumulative_alpha": [round(a, 5) for a in b["cumulative_alpha"]],
    "nominal_p_boundary": [round(p, 5) for p in b["nominal_p_boundaries"]],
    "z_boundary": [round(z, 3) for z in b["z_boundaries"]],
})

O'Brien-Fleming is conservative early (you need a very small p-value at the first look) and approaches the fixed-sample boundary at the end. Pocock spends more evenly.

In [ ]:
pocock = seq.group_sequential_boundaries(n_looks=5, alpha=0.05, spending="pocock")
fig, ax = plt.subplots(figsize=(8, 4))
t = b["information_times"]
ax.step(t, b["z_boundaries"], where="mid", label="O'Brien-Fleming", color="#4f46e5")
ax.step(t, pocock["z_boundaries"], where="mid", label="Pocock", color="#0ea5e9")
ax.axhline(1.96, ls="--", color="#dc2626", label="fixed-sample z=1.96")
ax.set_xlabel("information fraction"); ax.set_ylabel("critical z boundary")
ax.set_title("Sequential stopping boundaries"); ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "03_boundaries.png"), dpi=120, bbox_inches="tight"); plt.show()

## 2. Monitoring a real stream - naive p vs. always-valid p

In [ ]:
from src.data_pipeline.simulate import simulate_sequential_stream

stream = simulate_sequential_stream(n_total=20000, p_a=0.10, relative_lift=0.25, seed=11)
ctrl = stream.loc[stream.variant == "A", "converted"].values
trt = stream.loc[stream.variant == "B", "converted"].values

mon = seq.sequential_monitoring(ctrl, trt, n_looks=6, alpha=0.05,
                                spending="obrien-fleming", metric_type="binary")
pd.DataFrame(mon["looks"])[["look", "n_per_arm", "naive_p_value",
                            "nominal_p_boundary", "always_valid_p_value",
                            "crosses_boundary", "always_valid_significant"]]

In [ ]:
looks = pd.DataFrame(mon["looks"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(looks.n_per_arm, looks.naive_p_value, "o-", label="naive p-value", color="#94a3b8")
ax.plot(looks.n_per_arm, looks.always_valid_p_value, "s-", label="always-valid p-value", color="#4f46e5")
ax.plot(looks.n_per_arm, looks.nominal_p_boundary, "^--", label="alpha-spending boundary", color="#dc2626")
ax.set_yscale("log"); ax.set_xlabel("users per arm"); ax.set_ylabel("p-value (log scale)")
ax.set_title("Peek-safe monitoring"); ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "03_monitoring.png"), dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
sequential_conclusion(mon)

## Takeaways
* The naive p-value crosses 0.05 well before the always-valid p-value - that early "win" is exactly the false positive that peeking creates.
* Always-valid p-values let you check the dashboard as often as you like without inflating the error rate.
* Choose O'Brien-Fleming when early stopping matters little and power matters a lot; Pocock when you expect to stop early.